In [ ]:
%%html
<style>
toc {
    position: fixed;
    top: 50px;
    left: 20px;
    width: 250px;
    max-height: 80vh;
    overflow-y: auto;
    background: #f8f8f8;
    padding: 15px;
    border: 1px solid #ddd;
    border-radius: 5px;
    z-index: 1000;
}

</style>

<script>
// Auto-generate TOC from headers
document.addEventListener("DOMContentLoaded", function() {
    const headers = document.querySelectorAll('h1, h2, h3');
    const toc = document.getElementById('toc');
    
    if (toc && headers.length > 0) {
        let tocHTML = '<h3>Contents</h3><ul>';
        headers.forEach((header, index) => {
            const id = 'section-' + index;
            header.id = id;
            const level = parseInt(header.tagName[1]);
            const indent = (level - 1) * 20;
            tocHTML += `<li style="margin-left: ${indent}px">
                <a href="#${id}">${header.textContent}</a>
            </li>`;
        });
        tocHTML += '</ul>';
        toc.innerHTML = tocHTML;
    }
});
</script>

# Table of Contents
<div id="toc"></div>

Setting up parameter block for papermill

In [ ]:
# parameter defaults for testing - overwritten by papermill
SUBSET_MODE = "DICT"
SUBSET_MODE_CUSTOM_DICT = dict()

FILE_NAME = "Plasma_Cells_bulkRNA"
ADDITIONAL_SUBDIR = "Plasma_Cells/"
ANALYSIS_DESC = "ULM Scoring across Bulk Plasma Cells, per Sample"
COMPARTMENT_DESCRIPTION = "Bulk Plasma Cells"

RELOAD_IF_EXISTING=True
# Example contents
#cellID_short=dict(include=["NK_adp","NK_CD56dim","NK_CD56bright","NK_resident"],exclude=None), 
#lineage_group=dict(include=None, exclude=["LQ"])

# Legacy parameter
CLUSTERS_ANALYSIS = ["NK_adp"]

CUSTOM_PATHWAYS = {
    "PC_1_UP": ["NUSAP1", "H3C8", "PRIM2", "TOP2A", "CDK1", "AURKB", "PLK1", "H3C10", "HMGB2", "H4C3", "TPX2", "H2BC5", "KPNA2", "LIG1", "TK1", "STMN1", "UBE2S", "TYMS", "LMNB1", "EZH2", "UBE2T", "HMGN2", "TUBA1B", "CKS1B", "TUBB4B", "TUBB", "H2AZ1", "TUBA1C", "H2AZ1"],
    # "PC_1_UP_2020A": ["HIST1H1B3", "NUSAP1", "HIST1H3G", "HMGN2", "TOP2A", "CDK1", "AURKB", "PLK1", "HIST1H3H", "HMGB2", "HIST1H4C6", "TPX2", "HIST1H2BD", "KPNA2", "LIG1", "TK1", "STMN1", "UBE2S", "TYMS", "LMNB1", "EZH2", "UBE2T", "HMGN2", "TUBA1B", "CKS1B", "TUBB4B", "TUBB", "H2AFZ", "TUBA1C", "H2AZ1"],
    "PC_2_UP": ["NEAT1", "NLRP1", "OGT", "PTEN", "DISC1", "SNRPD1", "CDH16", "RAS4A", "C9JAP3", "NDUFA3", "PTK2", "IRF3", "POP6", "VPS29", "SIGMAR1", "JAK3", "EI-BP1L1", "OXR1", "SDHA"],
    # "PC_2_UP_2020A": ["NEAT1", "NLRP1", "OGT", "PTEN", "DISC1", "SNRPD1", "CDH16", "RAS4A", "C9JAP3", "NDUFA3", "PTK2", "IRF3", "POP6", "VPS29", "SIGMAR1", "JAK3", "EIBP1L1", "OXR1", "SDHA"],
    "PC_2_Down": ["H3C2", "H3C8", "H3C10", "H4C3", "STMN1", "HMGB2", "TUBA1B", "TUBB", "H2AZ1", "PTG82", "DLL1", "PTK2", "UBE2S", "CHMP3", "CHMP3", "NDUFA3", "RAS4A", "SNRPD1", "SIGRP2", "IER5", "PTK2", "UBE2S", "SHSF10", "VPS29", "OGT", "DDX39A", "NEAT1", "PRPF3", "TUBB4B", "MDM4", "MAU2", "CDKN1B", "SIGMAR1", "SDHA", "UNK1", "OXR1"],
    # "PC_2_DOWN_2020A": ["HIST1H1B3", "HIST1H3G", "HIST1H3H", "HIST1H4C6", "STMN1", "HMGB2", "TUBA1B", "TUBB", "H2AFZ", "PTG82", "DLL1", "PTK2", "UBE2S", "CHMP3", "CHMP3", "NDUFA3", "RAS4A", "SNRPD1", "SIGRP2", "IER5", "PTK2", "UBE2S", "SHSF10", "VPS29", "OGT", "DDX39A", "NEAT1", "PRPF3", "TUBB4B", "MDM4", "MAU2", "CDKN1B", "SIGMAR1", "SDHA", "UNK1", "OXR1"],
    "PC_3_DOWN": ["H3C2", "H3C8", "H2BC5", "H3C10", "H4C3", "HMGB2", "DISC1", "PTEN", "SNRPD1", "SIGRP2", "IER5"],
    # "PC_3_DOWN_2020A": ["HIST1H1B3", "HIST1H3G", "HIST1H2BD", "HIST1H3H", "HIST1H4C6", "HMGB2", "DISC1", "PTEN", "SNRPD1", "SIGRP2", "IER5"]
}




Setting up output covariates and other things

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import gzip
from pathlib import Path


print(os.getcwd())
os.chdir('/workspaces/Junia_Collab_Spatial/')
print(os.getcwd())


# Load gene list
G_list = pd.read_csv(os.path.join("resources", "grch38_and_grch37_ensembl_allids.csv"))
G_list["hgnc_symbol"] = G_list["hgnc_symbol_GRCh38"]
# Load count matrix
BULK_file = "data/Bulk/COUNTS_Gene_Based_MMRF_CoMMpass_IA22_star_geneUnstranded_counts.tsv.gz"

# Read compressed TSV file with gene names as index
counts = pd.read_csv(BULK_file, sep='\t', index_col="Gene", compression='gzip')
counts = counts.iloc[:60623, :]  # remove qc lines
sample_list = list(counts.columns)

# Metadata file paths
visit_id_output_dir = "data/metadata_032525/all_commpass_per_visit/"
public_id_output_dir = "data/metadata_032525/all_commpass_per_pat/"
public_id_output_name = "032525_per_pat.rds"
visit_id_output_name = "032525_per_visit.rds"

sub_directory = f'{ADDITIONAL_SUBDIR}/{FILE_NAME}/' 

output_dir_fig_tables = "output/Junia_Spatial_Analyses/" + sub_directory
os.makedirs(output_dir_fig_tables, exist_ok=True)
# Load RDS files (assuming they're saved as pickle files or we need to use pyreadr)

import subprocess
import sys
import importlib

def install_and_import(package):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    finally:
        globals()[package] = importlib.import_module(package)
        
install_and_import("pyreadr")
visitMD = pyreadr.read_r(os.path.join(visit_id_output_dir, visit_id_output_name))[None]
patMD = pyreadr.read_r(os.path.join(public_id_output_dir, public_id_output_name))[None]


# Extract sample information using string slicing
BM_or_PB = [x[12:14] for x in sample_list]  # Python uses 0-based indexing
pos_or_neg = [x[20:23] for x in sample_list]
visit_id = [x[0:11] for x in sample_list]
public_id = [x[0:9] for x in sample_list]

# Create bulk metadata dataframe
md_bulk = pd.DataFrame({
    'bulk_id': sample_list,
    'public_id': public_id,
    'd_visit_specimen_id': visit_id,
    'tissue_type': BM_or_PB
})

# Filter for BM tissue type
md_bulk = md_bulk[md_bulk['tissue_type'] == 'BM']

# Left join with patient and visit metadata
md_bulk = md_bulk.merge(patMD, on='public_id', how='left')
md_bulk = md_bulk.merge(visitMD, on=['public_id', 'd_visit_specimen_id'], how='left')
md_bulk = md_bulk.set_index('bulk_id')


# Filter for baseline samples with non-missing AFR ancestry
md_bulk_tmp = md_bulk[(md_bulk['VJ_INTERVAL'] == 'Baseline') & 
                      (md_bulk['ANCESTRY.AFR'].notna())]

# Select counts for final filtered samples
counts_final = counts[md_bulk_tmp.index]




In [ ]:
# Display gene list structure to understand mapping columns
print("Gene list columns:", G_list.columns.tolist())
print("Gene list shape:", G_list.shape)
print("\nFirst few rows:")
print(G_list.head())

# Create mapping dictionaries for gene ID conversion
# HGNC symbol to Ensemble ID
hgnc_to_ensembl = dict(zip(G_list['hgnc_symbol'].fillna(''), G_list['ensembl_gene_id']))
hgnc_to_ensembl = {k: v for k, v in hgnc_to_ensembl.items() if k != ''}

# Ensemble ID to HGNC symbol
ensembl_to_hgnc = dict(zip(G_list['ensembl_gene_id'], G_list['hgnc_symbol'].fillna('')))

print(f"\nCreated mapping dictionaries:")
print(f"  HGNC to Ensembl: {len(hgnc_to_ensembl)} mappings")
print(f"  Ensembl to HGNC: {len(ensembl_to_hgnc)} mappings")

In [ ]:
md_bulk_tmp.head()

In [ ]:
counts_final.head()

In [ ]:
# Create a scanpy AnnData object from counts_final and md_bulk_tmp
import scanpy as sc
import anndata as ad
import decoupler as dc
import pickle
import os
# Create AnnData object
# counts_final should be genes x samples, so we need to transpose for scanpy (samples x genes)
adata = ad.AnnData(X=counts_final.T)

# Set gene names as var (variable/feature names)
adata.var_names = counts_final.index
adata.var_names_make_unique()

# Set sample names as obs_names (observation names)
adata.obs_names = counts_final.columns

# Add metadata from md_bulk_tmp to adata.obs
# Make sure the order matches
adata.obs = md_bulk_tmp.reindex(adata.obs_names)

# Display basic info about the AnnData object
print(f"AnnData object with n_obs × n_vars = {adata.n_obs} × {adata.n_vars}")
print(f"Sample metadata columns: {list(adata.obs.columns)}")
print(f"First 5 samples: {list(adata.obs_names[:5])}")
print(f"First 5 genes: {list(adata.var_names[:5])}")

adata

Author: William Pilcher, Emory University/Georgia Institute of Technology

This performs the following:
    1) Loads .h5ad data containing converted Seurat objects with counts in raw.X, and log-normalized data (cp10k) in .X
    2) Subsets to Baseline + the Cell Type of Interest
    3) Performs pseudobulking via the decoupler package, filtering rare genes and patients with fewer than 10 cells of the compartment of interest
    4) py-deseq to perform differential expression for AFR Ancestry
    5) py-deseq for LFC shrinkage for GSEA from hongwei
    6) Tests on DEG results with decoupler implementations of the target/receptor network, progeny, and hallmark.
    7) (After manually saving the notebook), exports notebook to .html

As part of an additional screening step, we will also quickly assess the outcomes of all markers just to see if any relationship exists between the strongly ancestry associated markers and patient outcomes. These do not necessarily reflect the final statistics for reporting survival.

Only the differentially expressed genes will be the 'final' result generated by this report. Pathway analysis will be subject to secondary assessment via GSEA. 

Final visualizations will be done in R. 

Printing time notebook was generated along with the container image version.

Note - the container image will only contain python analysis package and will not include notebook functionality or things like 'ipynbcompress'. These are installed in the devcontainer dockerfile.

<h1>Loading and initial preprocessing</h1>

In [ ]:
from datetime import datetime
import os


current_time = datetime.today().strftime("%Y-%m-%d;%H:%M:%S")

print(f'Notebook Run Time: {current_time}')
print(f'Container Version: {os.environ.get("CONTAINER_VERSION")}')


Loading the full object, and checking file contents

In [ ]:
# Just checking embeddings
adata.obsm

In [ ]:
adata.obs.head()

Ensuring pandas is treating the continuous covariate columns we will be using as numeric covariates

In [ ]:
# Post conversion - numeric covariates treated as factors. Ensure they are correctly treated as numeric for downstream analyses.
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ttcpfs','ttcos']

adata.obs[cont_cov_col] = adata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
adata.obs.dtypes

Subset to baseline + the cell types of interest

Creating other covariates and moving counts from raw -> layers

In [ ]:
# Seurat creates a .raw slot instead of using .layers for counts data. Copy the raw matrix to a layer, and delete the raw slot.
adata.layers['counts'] = adata.X


<h1> Decoupler Pseudobulking </h1>

In [ ]:
#papermill_description=Pseudobulking
pdata = adata.copy()

In [ ]:
# Ensuring the covariates are correctly numeric in the pseudobulked object
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ttcpfs','ttcos']

pdata.obs[cont_cov_col] = pdata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
pdata.obs.dtypes

Sample filtering 

Assessing for batch effect/variance in pseudobulked PCs

In [ ]:
# Store raw counts in layers
pdata.layers["counts"] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
pdata.layers["nolog_normalized"] = pdata.X.copy()
sc.pp.log1p(pdata)
pdata.layers["normalized"] = pdata.X.copy()
sc.pp.scale(pdata, max_value=10)
pdata.layers["scaled"] = pdata.X.copy()

sc.tl.pca(pdata)

# Return raw counts to X
dc.pp.swap_layer(adata=pdata, key="counts", inplace=True)

In [ ]:
pdata.obs.head()

In [ ]:
pdata.obsm.keys()

In [ ]:
dc.tl.rankby_obsm(pdata, key="X_pca", obs_keys=["d_pt_sex","d_dx_amm_age", "IMWG_2024_HR"])

In [ ]:
if(FILE_NAME!="Fibro"):
    sc.pl.pca_variance_ratio(pdata)
    dc.pl.obsm(adata=pdata, return_fig=True, nvar=7, titles=["PC scores", "Adjusted p-values"], figsize=(10, 5))
else:
    print("Err in Fibro class- skipping QC plots")

In [ ]:
sc.pl.pca(
    pdata,
    color=['d_pt_sex', 'IMWG_2024_HR', 'd_dx_amm_age'],
    ncols=2,
    size=300,
    frameon=True,
)

Filtering by gene expression: (higher threshold for bulk RNA)

In [ ]:
dc.pl.filter_by_expr(
    adata=pdata,
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)


In [ ]:
dc.pp.filter_by_expr(
    adata=pdata,
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)


In [ ]:
pdata_filt = pdata # Don't need to copy 

<h3> Custom Gene Sets </h3>

In [ ]:
#papermill_description=Decoupler_Custom

# Convert CUSTOM_PATHWAYS dictionary to pandas DataFrame
# Each row is a gene-pathway pair
# For bulk RNA-seq, we need to convert HGNC symbols to Ensemble IDs
pathway_data = []
unmapped_genes = []

for pathway_name, genes in CUSTOM_PATHWAYS.items():
    for gene in genes:
        # Try to map HGNC symbol to Ensemble ID
        ensemble_id = hgnc_to_ensembl.get(gene, None)
        
        if ensemble_id:
            pathway_data.append({
                'source': pathway_name, 
                'target': ensemble_id,
                'hgnc_symbol': gene
            })
        else:
            # Gene symbol not found - might already be Ensemble ID or unmapped
            # Check if it's already in the count matrix
            if gene in pdata_filt.var_names:
                pathway_data.append({
                    'source': pathway_name, 
                    'target': gene,
                    'hgnc_symbol': ensembl_to_hgnc.get(gene, gene)
                })
            else:
                unmapped_genes.append((pathway_name, gene))

custom_pathways_df = pd.DataFrame(pathway_data)

print(f"Created pathway dataframe with {len(custom_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {custom_pathways_df['source'].nunique()}")
print(f"Number of unique Ensemble IDs: {custom_pathways_df['target'].nunique()}")

if unmapped_genes:
    print(f"\nWarning: {len(unmapped_genes)} genes could not be mapped:")
    for pathway, gene in unmapped_genes[:10]:
        print(f"  {pathway}: {gene}")
    if len(unmapped_genes) > 10:
        print(f"  ... and {len(unmapped_genes) - 10} more")

# Keep only source and target for decoupler
custom_pathways_df_decoupler = custom_pathways_df[['source', 'target']].drop_duplicates(subset=['source','target'])

# Save mapping for later use
custom_pathways_mapping = custom_pathways_df[['target', 'hgnc_symbol']].drop_duplicates()

print(f"\nPathways for decoupler: {len(custom_pathways_df_decoupler)} pairs")
custom_pathways_df_decoupler.head()

ULM Based Approach

In [ ]:
dc.mt.ulm(data=pdata_filt, net=custom_pathways_df_decoupler, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
print(f"Saved custom pathways ULM scores to: {output_dir_fig_tables}{FILE_NAME}_custom_pathways_ulm_scores.csv")

custom_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
custom_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_custom_pathways_ulm_scores.csv")

<h1> Gene expression survival analyses </h1>

In [ ]:
#papermill_description=Survival

import sys
n_pat = pdata_filt.n_obs
if(n_pat < 23):
    print(f"Too few patient for survival analyses")
    sys.exit(0)
else:
    print(f"Survival analyses can continue")


In [ ]:
# Convert survival columns to float to avoid categorical issues
pdata_filt.obs['ttcpfs'] = pd.to_numeric(pdata.obs['ttcpfs'], errors='coerce')
pdata_filt.obs['censpfs'] = pd.to_numeric(pdata.obs['censpfs'], errors='coerce')

print("Converted survival columns to numeric:")
print(f"ttcpfs dtype: {pdata_filt.obs['ttcpfs'].dtype}")
print(f"censpfs dtype: {pdata_filt.obs['censpfs'].dtype}")
print(f"ttcpfs range: {pdata_filt.obs['ttcpfs'].min():.1f} - {pdata_filt.obs['ttcpfs'].max():.1f}")
print(f"censpfs values: {pdata_filt.obs['censpfs'].unique()}")

In [ ]:
#papermill_description=COX_model_age

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_site_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_age_only_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_site_results_file):
    print(f"Loading existing Cox regression (age model) results from: {cox_site_results_file}")
    test_results_site = pd.read_csv(cox_site_results_file, index_col=0)
    print(f"Loaded {len(test_results_site)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (age adjustment) ===")
    test_results_site = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['d_dx_amm_age'],
        max_genes=None,
        verbose=True
    )
    
    # Map Ensemble IDs to HGNC symbols
    test_results_site['ensembl_gene_id'] = test_results_site['gene']
    test_results_site['hgnc_symbol'] = test_results_site['ensembl_gene_id'].map(lambda x: ensembl_to_hgnc.get(x, ''))
    test_results_site['gene_display'] = test_results_site.apply(
        lambda row: row['hgnc_symbol'] if row['hgnc_symbol'] else row['ensembl_gene_id'], 
        axis=1
    )
    
    test_results_site["model"] = "d_dx_amm_age"
    test_results_site["survival_type"] = "PFS"
    test_results_site.to_csv(cox_site_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_site[test_results_site['status'] == 'success'].nsmallest(30, 'p_value')
display_cols = ['gene_display', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']
if 'gene_display' in test_results_site.columns:
    print(top_results[display_cols])
else:
    print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_site['p_value'] < 0.05).any():
    # Use gene_display for plotting if available
    plot_df = test_results_site.copy()
    if 'gene_display' in plot_df.columns:
        plot_df['gene'] = plot_df['gene_display']
    
    wp_scanpy.plot_cox_volcano(plot_df, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (age Adjusted) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_site['fdr_p_value'] < 0.05).any():
    plot_df = test_results_site.copy()
    if 'gene_display' in plot_df.columns:
        plot_df['gene'] = plot_df['gene_display']
    
    wp_scanpy.plot_cox_volcano(plot_df, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (age Adjusted) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")

In [ ]:
#papermill_description=COX_model_full

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_full_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_full_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_full_results_file):
    print(f"Loading existing Cox regression (full model) results from: {cox_full_results_file}")
    test_results_full = pd.read_csv(cox_full_results_file, index_col=0)
    print(f"Loaded {len(test_results_full)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (Full Model) ===")
    test_results_full = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['d_dx_amm_age','d_amm_tx_asct_1st', 'd_pt_sex'],
        max_genes=None,
        verbose=True
    )
    
    # Map Ensemble IDs to HGNC symbols
    test_results_full['ensembl_gene_id'] = test_results_full['gene']
    test_results_full['hgnc_symbol'] = test_results_full['ensembl_gene_id'].map(lambda x: ensembl_to_hgnc.get(x, ''))
    test_results_full['gene_display'] = test_results_full.apply(
        lambda row: row['hgnc_symbol'] if row['hgnc_symbol'] else row['ensembl_gene_id'], 
        axis=1
    )
    
    test_results_full["model"] = "d_dx_amm_age + d_amm_tx_asct_1st + d_pt_sex"
    test_results_full["survival_type"] = "PFS"
    test_results_full.to_csv(cox_full_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_full[test_results_full['status'] == 'success'].nsmallest(30, 'p_value')
display_cols = ['gene_display', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']
if 'gene_display' in test_results_full.columns:
    print(top_results[display_cols])
else:
    print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_full['p_value'] < 0.05).any():
    # Use gene_display for plotting if available
    plot_df = test_results_full.copy()
    if 'gene_display' in plot_df.columns:
        plot_df['gene'] = plot_df['gene_display']
    
    wp_scanpy.plot_cox_volcano(plot_df, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_full['fdr_p_value'] < 0.05).any():
    plot_df = test_results_full.copy()
    if 'gene_display' in plot_df.columns:
        plot_df['gene'] = plot_df['gene_display']
    
    wp_scanpy.plot_cox_volcano(plot_df, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")

Test for Outcome Associations

<h3> Outcome Associated Pathways via Decoupler </h3>

Repeating the decoupler analyses, but basically doing it by sign(log(HR)) * -log(p-val) from the adjusted coxPH model.

In [ ]:
# Use ensemble_gene_id for indexing (matches the count matrix)
if 'ensembl_gene_id' in test_results_full.columns:
    test_results_full = test_results_full.set_index('ensembl_gene_id')
else:
    test_results_full = test_results_full.set_index('gene')
test_results_full

In [ ]:
test_results_full[["hazard_ratio_log10"]] = test_results_full[["hazard_ratio"]].apply(np.log2)

test_results_full["signed_pval"] = test_results_full["p_value"].apply(np.log2).multiply(-1).multiply(test_results_full["hazard_ratio_log10"].apply(np.sign))


data = test_results_full[["p_value"]].T.rename(index={"p_value": "AFRHigh_vs_AFRLow"}).apply(np.log2).multiply(-1)
data

dataHR = test_results_full[["hazard_ratio"]].T.rename(index={"hazard_ratio": "AFRHigh_vs_AFRLow"}).apply(np.log2)
dataHR

test_results_full


In [ ]:
data_mult = data.multiply(dataHR.apply(np.sign))

data_mult

<h3> CoxPH - Custom Pathways </h3>

In [ ]:
# Run - use the Ensemble ID version of the pathway dataframe
custom_acts, custom_padj = dc.mt.ulm(data=data_mult, net=custom_pathways_df_decoupler)

# Filter by sign padj
msk = (custom_padj.T < 0.05).iloc[:, 0]
custom_acts = custom_acts.loc[:, msk]

custom_acts

In [ ]:
custom_pathways_df_decoupler

In [ ]:

if(len(custom_acts.columns) > 0):
    dc.pl.barplot(data=custom_acts, name="AFRHigh_vs_AFRLow", figsize=(6, 3))
else:
    print("No significant hallmark pathways found")

if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=custom_pathways_df_decoupler,
            name=pathway,
        )
        # Map leading edge Ensemble IDs to HGNC symbols for display
        le_display = [ensembl_to_hgnc.get(gene, gene) for gene in le[:10]]
        print(f"{pathway} leading edge:", le_display)
else:
    print("No significant custom pathways found")


if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        # Create a version of test_results_full with gene_display as index for volcano plot
        test_results_display = test_results_full.copy()
        if 'gene_display' in test_results_display.columns:
            test_results_display.index = test_results_display['gene_display']
        
        # Create a version of custom_pathways_df_decoupler with gene_display targets
        custom_pathways_display = custom_pathways_df_decoupler.copy()
        custom_pathways_display['target'] = custom_pathways_display['target'].map(
            lambda x: ensembl_to_hgnc.get(x, x)
        )
        
        dc.pl.volcano(
            data=test_results_display,
            x="hazard_ratio_log10",
            y="p_value",
            net=custom_pathways_display,
            name=pathway,
            top=50,
            thr_stat=0.08,
            thr_sign=0.05
        )
else:
    print("No significant custom pathways found")

<h1>ULM Pathway Survival Analysis</h1>

Perform comprehensive survival analysis on all ULM pathway scores:
1. Cox regression with z-scored pathway scores
2. Forest plots visualizing hazard ratios
3. Kaplan-Meier curves with multiple stratification methods
4. Optimal cutpoint analysis with HR curves

## Cox Regression: Study Site Only

First analysis: adjusting only for age

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
pdata_filt.obs['bulk_id'] = pdata_filt.obs_names

In [ ]:
#papermill_description=ULM_COX_age
import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
ulm_cox_site_results_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_age_only_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(ulm_cox_site_results_file):
    print(f"Loading existing ULM Cox regression (age model) results from: {ulm_cox_site_results_file}")
    ulm_results_site = pd.read_csv(ulm_cox_site_results_file, index_col=0)
    print(f"Loaded {len(ulm_results_site)} pathways from existing results")
else:
    print("=== Running Cox regression analysis with all ULM pathways (age adjustment) ===")
    ulm_results_site = wp_scanpy.ulm_pathway_cox_analysis(
        pdata_filt, 
        ulm_obsm_key='score_ulm',
        covariates=['d_dx_amm_age'],
        sample_id='bulk_id',
        verbose=True
    )
    
    ulm_results_site["model"] = "d_dx_amm_age"
    ulm_results_site["survival_type"] = "PFS"
    ulm_results_site.to_csv(ulm_cox_site_results_file)

# Display results
print("\nTop 10 pathways (by p-value):")
top_results = ulm_results_site[ulm_results_site['status'] == 'success'].nsmallest(10, 'p_value')
print(top_results[['pathway', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

In [ ]:
# Plot forest plot for age model
if (ulm_results_site['p_value'] < 0.05).any():
    wp_scanpy.plot_pathway_forest_plot(
        ulm_results_site,
        title=f"{COMPARTMENT_DESCRIPTION}: ULM Pathway Cox Regression (age Adjusted)",
        p_threshold=0.05,
        top_n=30,
        figsize=(10, 12)
    )
else:
    print("No significant pathways found at p < 0.05 - Skipping Forest Plot")

## Cox Regression: Fully Adjusted Model

Second analysis: adjusting for age, ASCT status, sex

In [ ]:
#papermill_description=ULM_COX_fully_adjusted

# Check if results already exist and should be reloaded
ulm_cox_full_results_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_fully_adjusted_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(ulm_cox_full_results_file):
    print(f"Loading existing ULM Cox regression (Fully Adjusted model) results from: {ulm_cox_full_results_file}")
    ulm_results_full = pd.read_csv(ulm_cox_full_results_file, index_col=0)
    print(f"Loaded {len(ulm_results_full)} pathways from existing results")
else:
    print("=== Running Cox regression analysis with all ULM pathways (Fully Adjusted) ===")
    ulm_results_full = wp_scanpy.ulm_pathway_cox_analysis(
        pdata_filt, 
        ulm_obsm_key='score_ulm',
        covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex'],
        sample_id='bulk_id',
        verbose=True
    )
    
    ulm_results_full["model"] = "Fully_Adjusted"
    ulm_results_full["survival_type"] = "PFS"
    ulm_results_full.to_csv(ulm_cox_full_results_file)

# Display results
print("\nTop 10 pathways (by p-value):")
top_results = ulm_results_full[ulm_results_full['status'] == 'success'].nsmallest(10, 'p_value')
print(top_results[['pathway', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

In [ ]:
# Plot forest plot for Fully Adjusted model

wp_scanpy.plot_pathway_forest_plot(
    ulm_results_full,
    title=f"{COMPARTMENT_DESCRIPTION}: ULM Pathway Cox Regression (Fully Adjusted)",
    p_threshold=0.05,
    top_n=30,
    figsize=(10, 12)
)


## Combined Results

Merge results from both models into a single summary table

In [ ]:
# Combine results from both models into a wide format summary
ulm_combined_results = pd.merge(
    ulm_results_site[['pathway', 'hazard_ratio', 'hr_lower_ci', 'hr_upper_ci', 'p_value', 'fdr_p_value']],
    ulm_results_full[['pathway', 'hazard_ratio', 'hr_lower_ci', 'hr_upper_ci', 'p_value', 'fdr_p_value']],
    on='pathway',
    suffixes=('_site_only', '_fully_adjusted')
)

# Save combined results
combined_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_combined_results.csv"
ulm_combined_results.to_csv(combined_file, index=False)
print(f"Saved combined ULM Cox regression results to: {combined_file}")

# Display top results from both models
print("\nTop 10 pathways by age model p-value:")
display(ulm_combined_results.nsmallest(10, 'p_value_site_only')[
    ['pathway', 'hazard_ratio_site_only', 'p_value_site_only', 
     'hazard_ratio_fully_adjusted', 'p_value_fully_adjusted']
])

print("\nTop 10 pathways by Fully Adjusted model p-value:")
display(ulm_combined_results.nsmallest(10, 'p_value_fully_adjusted')[
    ['pathway', 'hazard_ratio_site_only', 'p_value_site_only', 
     'hazard_ratio_fully_adjusted', 'p_value_fully_adjusted']
])

## Kaplan-Meier Curves for Top Pathways

For each significant pathway, generate KM curves with different stratification methods:
1. Median cutpoint
2. Quartile split (4 groups)
3. Optimal cutpoint (with HR curve showing all tested cutpoints)

In [ ]:
# Identify top pathways to plot (based on fully adjusted model)
# We'll plot KM curves for pathways with p < 0.05
significant_pathways = ulm_results_full[
    (ulm_results_full['status'] == 'success') & 
    (ulm_results_full['p_value'] < 0.05)
].sort_values('p_value')

print(f"Found {len(significant_pathways)} significant pathways (p < 0.05) to plot KM curves")
print("\nTop 10 significant pathways:")
print(significant_pathways.head(10)[['pathway', 'hazard_ratio', 'p_value']])

# Select top pathways to plot (limit to top 10 to avoid too many plots)
pathways_to_plot = ulm_results_full['pathway'].tolist() #significant_pathways.head(10)['pathway'].tolist()
print(f"\nWill generate KM curves for {len(pathways_to_plot)} pathways")

### KM Curves: Median Cutpoint

Split patients into high/low groups based on median pathway score

In [ ]:
# Generate KM curves with median cutpoint for each significant pathway
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='median',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex'],
            figsize=(10, 6)
        )
    except Exception as e:
        print(f"Error plotting KM curve for {pathway}: {str(e)}")

### KM Curves: Quartile Split

Split patients into 4 groups (Q1-Q4) based on pathway score quartiles

In [ ]:
# Generate KM curves with quartile split for each significant pathway
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='quartiles',
            covariates=None,  # Quartile comparison typically doesn't use Cox adjustment
            figsize=(10, 6)
        )
    except Exception as e:
        print(f"Error plotting KM curve for {pathway}: {str(e)}")

### KM Curves: Optimal Cutpoint

Find optimal cutpoint (between 25th-75th percentile) that minimizes p-value.
Also display HR curve showing hazard ratios at each tested cutpoint.

In [ ]:
# Generate optimal cutpoint analysis for each significant pathway
# This includes both the HR curve and KM curve
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        # First, plot the HR curve showing all tested cutpoints
        print("\n--- HR Curve (showing optimal cutpoint) ---")
        cutpoint_results, optimal_cutpoint = wp_scanpy.plot_optimal_cutpoint_hr_curve(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex'],
            figsize=(10, 5)
        )
        
        # Then plot the KM curve using the optimal cutpoint
        print("\n--- KM Curve (using optimal cutpoint) ---")
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='optimal',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex'],
            figsize=(10, 6)
        )
        
    except Exception as e:
        print(f"Error in optimal cutpoint analysis for {pathway}: {str(e)}")